# Seminar 8: Regression vs Ranking Losses. Content ALS

## Goals

In this seminar we will:
1. Compare **regression** (MSE, WMSE) and **ranking** (BPR) loss functions for matrix factorization
2. Implement **BPR-MF** (Bayesian Personalized Ranking) from scratch with SGD
3. Derive and implement **Content ALS** — incorporating item content embeddings into ALS for cold-start
4. Evaluate all approaches on the **VK-LSVD** short-video dataset, including **cold-start** items

In [ ]:
!pip install implicit lightfm huggingface_hub polars

In [ ]:
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
from scipy import sparse
import polars as pl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

plt.style.use("ggplot")

---

## 1. VK-LSVD Dataset

We reuse the **VK-LSVD** dataset from Seminar 5. Key facts:

- **40B** interactions with rich feedback (`timespent`, `like`, `dislike`, `share`, `bookmark`, `click_on_author`, `open_comments`)
- **20M** short videos with **64-dimensional content embeddings** (learned from video content, float16)
- We use the **`up0.001_ip0.001`** subsample (~10K users, ~20K items)
- Temporal split: weeks 0–24 for training, week 25 for validation

Dataset: https://huggingface.co/datasets/deepvk/VK-LSVD

In [ ]:
SUBSAMPLE = "up0.001_ip0.001"
DATA_DIR = Path("VK-LSVD")

train_week_files = [
    f"subsamples/{SUBSAMPLE}/train/week_{i:02d}.parquet" for i in range(25)
]
val_week_files = [f"subsamples/{SUBSAMPLE}/validation/week_25.parquet"]
metadata_files = [
    "metadata/users_metadata.parquet",
    "metadata/items_metadata.parquet",
    "metadata/item_embeddings.npz",
]

for f in tqdm(train_week_files + val_week_files + metadata_files, desc="Downloading"):
    hf_hub_download(
        repo_id="deepvk/VK-LSVD",
        repo_type="dataset",
        filename=f,
        local_dir=str(DATA_DIR),
    )
print("Download complete.")

In [ ]:
train_weeks_dfs = []
for i in tqdm(range(25), desc="Loading train weeks"):
    path = DATA_DIR / f"subsamples/{SUBSAMPLE}/train/week_{i:02d}.parquet"
    wdf = pl.read_parquet(path)
    wdf = wdf.with_columns(pl.lit(i).cast(pl.UInt8).alias("week"))
    train_weeks_dfs.append(wdf)

train_df = pl.concat(train_weeks_dfs)
val_df = pl.read_parquet(
    DATA_DIR / f"subsamples/{SUBSAMPLE}/validation/week_25.parquet"
)

print(f"Train interactions: {len(train_df):,}")
print(f"Val interactions:   {len(val_df):,}")
print(f"Train users:        {train_df['user_id'].n_unique():,}")
print(f"Train items:        {train_df['item_id'].n_unique():,}")
train_df.head()

In [ ]:
items_meta = pl.read_parquet(DATA_DIR / "metadata/items_metadata.parquet")

train_item_set = set(train_df["item_id"].unique().to_list())
items_meta = items_meta.filter(pl.col("item_id").is_in(train_item_set))

print(f"Items metadata: {len(items_meta):,} rows")
print(f"Columns: {items_meta.columns}")
items_meta.head()

### Loading content embeddings

VK-LSVD provides **64-dimensional content embeddings** for each item, learned from video content. The dimensions are ordered by importance, so you can use any prefix (e.g., first 32 dims) as a lower-dimensional representation.

In [ ]:
emb_data = np.load(DATA_DIR / "metadata/item_embeddings.npz")
emb_item_ids = emb_data["item_id"]       # shape (N_total,)
emb_vectors = emb_data["embedding"]       # shape (N_total, 64), float16

print(f"Total embeddings: {emb_vectors.shape}")
print(f"Dtype: {emb_vectors.dtype}")
print(f"Embedding dim: {emb_vectors.shape[1]}")

### Building interaction matrices and ID mappings

In a short-video feed, **every impression is recorded** — including 1-second swipe-pasts. Rather than filtering these out, we use them as **weak negatives** via the iALS confidence weighting: items watched longer get higher confidence, while brief views have low confidence but still contribute to learning.

In [ ]:
all_user_ids = np.sort(train_df["user_id"].unique().to_numpy())
all_item_ids = np.sort(train_df["item_id"].unique().to_numpy())
n_users = len(all_user_ids)
n_items = len(all_item_ids)

print(f"n_users={n_users:,}, n_items={n_items:,}")


def map_ids(ids, sorted_all):
    """Map raw IDs to contiguous indices via binary search."""
    return np.searchsorted(sorted_all, ids)


user_idx_all = map_ids(train_df["user_id"].to_numpy(), all_user_ids)
item_idx_all = map_ids(train_df["item_id"].to_numpy(), all_item_ids)

# Binary interaction matrix (all impressions)
R_train = sparse.csr_matrix(
    (np.ones(len(train_df), dtype=np.float32),
     (user_idx_all, item_idx_all)),
    shape=(n_users, n_items),
)

# Timespent-weighted matrix (for confidence weighting in iALS)
ts_vals = train_df["timespent"].to_numpy().astype(np.float32)
R_timespent = sparse.csr_matrix(
    (ts_vals, (user_idx_all, item_idx_all)),
    shape=(n_users, n_items),
)

# Log-timespent weighted matrix (dampens extreme values)
R_log_ts = R_timespent.copy()
R_log_ts.data = np.log1p(R_log_ts.data).astype(np.float32)

density = R_train.nnz / (n_users * n_items)
print(f"R_train (binary):  {R_train.shape}, nnz={R_train.nnz:,}, density={density:.2%}")
print(f"R_timespent:       {R_timespent.shape}, nnz={R_timespent.nnz:,}")
print(f"R_log_ts:          {R_log_ts.shape}, nnz={R_log_ts.nnz:,}")

In [ ]:
# Build item content embedding matrix aligned with our item indices
# Filter to items present in our subsample
emb_mask = np.isin(emb_item_ids, all_item_ids)
sub_emb_ids = emb_item_ids[emb_mask]
sub_emb_vecs = emb_vectors[emb_mask].astype(np.float32)

# Map to internal indices
sub_emb_idx = map_ids(sub_emb_ids, all_item_ids)

# Create a (n_items, 64) embedding matrix; items without embeddings get zeros
EMB_DIM = sub_emb_vecs.shape[1]
item_content_emb = np.zeros((n_items, EMB_DIM), dtype=np.float32)
item_content_emb[sub_emb_idx] = sub_emb_vecs

has_embedding = np.zeros(n_items, dtype=bool)
has_embedding[sub_emb_idx] = True

print(f"Items with content embeddings: {has_embedding.sum():,} / {n_items:,}")
print(f"Embedding matrix shape: {item_content_emb.shape}")

### Evaluation setup

We use the same evaluation protocol as in Seminar 5: train on weeks 0–24, evaluate on week 25. Metrics: **HitRate@K**, **MRR@K**, **NDCG@K**.

In [ ]:
def evaluate(user_factors, item_factors, R_filter, val_user_items, k=10):
    """Compute HitRate@K, MRR@K, NDCG@K.

    R_filter: CSR matrix used to filter out seen items during scoring.
    val_user_items: dict[user_idx -> list[item_idx]] ground truth.
    """
    hits = 0
    mrr_sum = 0.0
    ndcg_sum = 0.0
    n_eval = 0

    for u, true_items in val_user_items.items():
        scores = item_factors @ user_factors[u]

        s, e = R_filter.indptr[u], R_filter.indptr[u + 1]
        scores[R_filter.indices[s:e]] = -np.inf

        n_score = len(scores)
        if k >= n_score:
            top_k = np.argsort(-scores)
        else:
            top_k = np.argpartition(-scores, k)[:k]
            top_k = top_k[np.argsort(-scores[top_k])]

        true_set = set(true_items)

        hits += int(bool(true_set & set(top_k)))

        for rank, item in enumerate(top_k, 1):
            if item in true_set:
                mrr_sum += 1.0 / rank
                break

        dcg = sum(
            1.0 / np.log2(r + 2)
            for r, item in enumerate(top_k)
            if item in true_set
        )
        n_rel = min(k, len(true_items))
        idcg = sum(1.0 / np.log2(i + 2) for i in range(n_rel))
        ndcg_sum += dcg / idcg if idcg > 0 else 0.0

        n_eval += 1

    if n_eval == 0:
        return {f"HitRate@{k}": 0, f"MRR@{k}": 0, f"NDCG@{k}": 0}
    return {
        f"HitRate@{k}": hits / n_eval,
        f"MRR@{k}": mrr_sum / n_eval,
        f"NDCG@{k}": ndcg_sum / n_eval,
    }

In [ ]:
# Build validation ground truth (all val interactions, not filtered by watch time)
val_uid_np = val_df["user_id"].to_numpy()
val_iid_np = val_df["item_id"].to_numpy()
val_in_train = np.isin(val_uid_np, all_user_ids) & np.isin(val_iid_np, all_item_ids)

val_u_idx = map_ids(val_uid_np[val_in_train], all_user_ids)
val_i_idx = map_ids(val_iid_np[val_in_train], all_item_ids)

val_user_items = defaultdict(list)
for u, i in zip(val_u_idx, val_i_idx):
    val_user_items[u].append(i)
val_user_items = dict(val_user_items)

print(f"Validation users (in train): {len(val_user_items):,}")
print(f"Avg val items per user: {np.mean([len(v) for v in val_user_items.values()]):.1f}")

K = 10

---

## 2. Regression losses for matrix factorization

In Seminars 4 and 5 we derived and implemented ALS with **squared-error** objectives. Let's recap and compare variants.

### 2.1 Explicit ALS (MSE)

For observed ratings $r_{ui}$, minimize:

$$
\mathcal{L}_{\text{MSE}} = \sum_{(u,i) \in \Omega} (r_{ui} - \mathbf{u}_u^\top \mathbf{v}_i)^2 + \lambda (\|\mathbf{u}_u\|^2 + \|\mathbf{v}_i\|^2)
$$

This is a **pointwise regression** loss: we treat each observed rating as a regression target and minimize prediction error. The ALS update for user $u$ is the familiar ridge regression solution:

$$
\mathbf{u}_u = (V_u^\top V_u + \lambda I)^{-1} V_u^\top \mathbf{r}_u
$$

### 2.2 Implicit ALS — iALS (Hu, Koren, Volinsky, 2008)

For implicit feedback, we don't have explicit ratings. Instead we define:
- **Preference**: $p_{ui} = \mathbb{1}[r_{ui} > 0]$ — did the user interact with item $i$?
- **Confidence**: $c_{ui} = 1 + \alpha \cdot r_{ui}$ — how confident are we in this preference?

The objective becomes a **weighted MSE over ALL user-item pairs** (not just observed):

$$
\mathcal{L}_{\text{iALS}} = \sum_{u,i} c_{ui} (p_{ui} - \mathbf{u}_u^\top \mathbf{v}_i)^2 + \lambda (\|\mathbf{u}_u\|^2 + \|\mathbf{v}_i\|^2)
$$

The key difference from explicit ALS:
- We sum over **all** entries, not just observed ones
- Unobserved entries are treated as **weak negatives** ($p_{ui}=0$) with low confidence ($c_{ui}=1$)
- Observed entries have **high confidence** ($c_{ui} = 1 + \alpha \cdot r_{ui}$)

The iALS update uses the efficient trick (see Seminar 5):

$$
\mathbf{u}_u = \left(V^\top V + V^\top (C_u - I) V + \lambda I\right)^{-1} V^\top C_u \mathbf{p}_u
$$

where $C_u = \text{diag}(c_{u1}, \ldots, c_{un})$ and $V^\top V$ is precomputed once per iteration.

### 2.3 Confidence functions

The choice of confidence function $c_{ui} = f(r_{ui})$ significantly affects iALS performance:

| Name | Formula | Intuition |
|------|---------|-----------|
| Linear | $c_{ui} = 1 + \alpha \cdot r_{ui}$ | Original Hu et al. — proportional to signal strength |
| Log | $c_{ui} = 1 + \alpha \cdot \log(1 + r_{ui})$ | Dampens extreme values (e.g., very long watch times) |
| Binary | $c_{ui} = 1 + \alpha \cdot \mathbb{1}[r_{ui} > 0]$ | Ignores signal magnitude, only presence/absence |

### The regression loss limitation

All these losses optimize **pointwise prediction accuracy**: how well can we predict whether/how much a user will interact with an item? But our actual goal is **ranking**: we want to place relevant items above irrelevant ones. A model can have low MSE but poor ranking quality — for example, if it systematically predicts scores of 0.49 for relevant items and 0.51 for irrelevant ones.

### iALS baseline via `implicit` library

We train the `implicit` library's ALS as our regression baseline, then compare confidence functions.

In [ ]:
from implicit.als import AlternatingLeastSquares

N_FACTORS = 64
REG = 0.01
N_ITERS = 15

# --- Baseline: binary interactions ---
ials_binary = AlternatingLeastSquares(
    factors=N_FACTORS, regularization=REG, iterations=N_ITERS, random_state=42
)
ials_binary.fit(R_train)

metrics_binary = evaluate(
    ials_binary.user_factors, ials_binary.item_factors,
    R_train, val_user_items, k=K
)
print("iALS (binary):", metrics_binary)

In [ ]:
# --- Confidence function comparison ---

# Linear confidence: c = 1 + alpha * timespent
ials_linear = AlternatingLeastSquares(
    factors=N_FACTORS, regularization=REG, iterations=N_ITERS, random_state=42
)
ials_linear.fit(R_timespent)

metrics_linear = evaluate(
    ials_linear.user_factors, ials_linear.item_factors,
    R_train, val_user_items, k=K
)
print("iALS (timespent conf.):", metrics_linear)

# Log confidence: c = 1 + alpha * log(1 + timespent)
ials_log = AlternatingLeastSquares(
    factors=N_FACTORS, regularization=REG, iterations=N_ITERS, random_state=42
)
ials_log.fit(R_log_ts)

metrics_log = evaluate(
    ials_log.user_factors, ials_log.item_factors,
    R_train, val_user_items, k=K
)
print("iALS (log conf.):      ", metrics_log)

In [ ]:
import pandas as pd

regression_results = pd.DataFrame([
    {"Model": "iALS (binary)", **metrics_binary},
    {"Model": "iALS (timespent conf.)", **metrics_linear},
    {"Model": "iALS (log conf.)", **metrics_log},
])
print(regression_results.to_string(index=False))

---

## 3. Ranking losses for matrix factorization

### 3.1 BPR — Bayesian Personalized Ranking (Rendle et al., 2009)

Instead of predicting absolute scores, BPR directly optimizes the **pairwise ordering** of items. The core idea: for each user $u$, an item $i$ that the user interacted with should be ranked **higher** than an item $j$ that the user did not interact with.

#### BPR-OPT criterion

Given a user $u$, a **positive** item $i$ (observed interaction) and a **negative** item $j$ (no interaction), define the preference score difference:

$$
\hat{x}_{uij} = \hat{r}_{ui} - \hat{r}_{uj} = \mathbf{u}_u^\top \mathbf{v}_i - \mathbf{u}_u^\top \mathbf{v}_j = \mathbf{u}_u^\top (\mathbf{v}_i - \mathbf{v}_j)
$$

BPR maximizes the log-likelihood of correct pairwise orderings:

$$
\mathcal{L}_{\text{BPR}} = \sum_{(u, i, j) \in D_S} \ln \sigma(\hat{x}_{uij}) - \lambda \left(\|\mathbf{u}_u\|^2 + \|\mathbf{v}_i\|^2 + \|\mathbf{v}_j\|^2\right)
$$

where $\sigma(x) = \frac{1}{1+e^{-x}}$ is the sigmoid function and $D_S = \{(u, i, j) \mid i \in \mathcal{I}_u^+, \; j \notin \mathcal{I}_u^+\}$ is the set of all valid triples.

#### SGD updates

We optimize via stochastic gradient descent. For a sampled triple $(u, i, j)$:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{u}_u} = (1 - \sigma(\hat{x}_{uij}))(\mathbf{v}_i - \mathbf{v}_j) - \lambda \mathbf{u}_u
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{v}_i} = (1 - \sigma(\hat{x}_{uij})) \mathbf{u}_u - \lambda \mathbf{v}_i
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{v}_j} = -(1 - \sigma(\hat{x}_{uij})) \mathbf{u}_u - \lambda \mathbf{v}_j
$$

At each step we sample a triple $(u, i, j)$, compute $\hat{x}_{uij}$, and update $\mathbf{u}_u, \mathbf{v}_i, \mathbf{v}_j$ in the direction of the gradient.

#### Why BPR can outperform regression losses

- BPR directly optimizes **pairwise ranking**, which aligns with ranking metrics (NDCG, MRR)
- Regression losses waste capacity predicting exact scores for items the user will never see
- BPR focuses model capacity on the **decision boundary** between relevant and irrelevant items

### 3.2 WARP — Weighted Approximate-Rank Pairwise loss (brief note)

WARP (Weston et al., 2011) extends the pairwise idea by **sampling violating negatives**: it keeps sampling negative items until it finds one that the model currently ranks above the positive. The loss is weighted by the approximate rank of the positive item — items ranked very low get a larger gradient. This makes WARP harder to implement but often outperforms BPR, especially for top-heavy metrics like NDCG@K.

We won't implement WARP from scratch here, but the `implicit` library provides it.

### 3.3 BPR-MF implementation from scratch

In [ ]:
class BPRMF:
    """BPR-MF: Matrix Factorization with Bayesian Personalized Ranking loss."""

    def __init__(self, n_factors=64, lr=0.01, reg=0.01,
                 n_epochs=10, n_samples_per_epoch=None, random_state=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.n_samples_per_epoch = n_samples_per_epoch
        self.random_state = random_state

        self.user_factors = None
        self.item_factors = None

    def fit(self, R_csr, verbose=True):
        """Fit BPR-MF on a CSR interaction matrix (binary or weighted, only nonzero entries matter)."""
        n_users, n_items = R_csr.shape
        rng = np.random.default_rng(self.random_state)

        # Initialize factors
        self.user_factors = rng.normal(0, 0.01, size=(n_users, self.n_factors)).astype(np.float32)
        self.item_factors = rng.normal(0, 0.01, size=(n_items, self.n_factors)).astype(np.float32)

        # Build per-user positive item sets for fast lookup
        user_pos_items = []
        for u in range(n_users):
            s, e = R_csr.indptr[u], R_csr.indptr[u + 1]
            user_pos_items.append(set(R_csr.indices[s:e].tolist()))

        # Collect all (user, pos_item) pairs for sampling
        users_arr, items_arr = R_csr.nonzero()
        n_interactions = len(users_arr)
        n_samples = self.n_samples_per_epoch or n_interactions

        losses = []

        for epoch in range(self.n_epochs):
            epoch_loss = 0.0

            # Shuffle interaction pairs
            perm = rng.integers(0, n_interactions, size=n_samples)

            for idx in tqdm(perm, desc=f"BPR epoch {epoch+1}/{self.n_epochs}",
                            leave=False, disable=not verbose):
                u = users_arr[idx]
                i = items_arr[idx]

                # Sample a negative item
                j = rng.integers(0, n_items)
                while j in user_pos_items[u]:
                    j = rng.integers(0, n_items)

                # Compute x_uij = u_u^T (v_i - v_j)
                u_vec = self.user_factors[u]
                v_i = self.item_factors[i]
                v_j = self.item_factors[j]

                x_uij = u_vec @ (v_i - v_j)

                # σ(-x_uij) = 1 / (1 + exp(x_uij))
                # This is the gradient multiplier for BPR: large when ordering is wrong
                x_clip = np.clip(x_uij, -500, 500)
                sig_neg = 1.0 / (1.0 + np.exp(x_clip))

                # SGD updates (gradient ascent on log-likelihood)
                self.user_factors[u] += self.lr * (sig_neg * (v_i - v_j) - self.reg * u_vec)
                self.item_factors[i] += self.lr * (sig_neg * u_vec - self.reg * v_i)
                self.item_factors[j] += self.lr * (-sig_neg * u_vec - self.reg * v_j)

                epoch_loss += np.log(1.0 - sig_neg + 1e-10)  # log σ(x_uij)

            avg_loss = epoch_loss / n_samples
            losses.append(avg_loss)
            if verbose:
                print(f"  Epoch {epoch+1}: avg BPR loss = {avg_loss:.4f}")

        return losses

### Sanity check on small data

In [ ]:
# Sanity check: BPR on small synthetic data
rng_test = np.random.default_rng(0)
R_small = sparse.random(50, 80, density=0.1, format="csr",
                        random_state=0, dtype=np.float32)
R_small.data[:] = 1.0

bpr_small = BPRMF(n_factors=8, lr=0.05, reg=0.001, n_epochs=5, random_state=0)
losses_small = bpr_small.fit(R_small, verbose=False)

print("Losses per epoch:", [f"{l:.4f}" for l in losses_small])
assert losses_small[-1] > losses_small[0], "BPR loss should increase (we maximize log σ(x_uij))"
print("Shape check: U =", bpr_small.user_factors.shape, "V =", bpr_small.item_factors.shape)
print("Sanity check passed.")

### Train BPR-MF on VK-LSVD

We use a subsample of interactions per epoch to keep runtime manageable.

In [ ]:
bpr_model = BPRMF(
    n_factors=N_FACTORS,
    lr=0.01,
    reg=0.001,
    n_epochs=15,
    n_samples_per_epoch=1_000_000,
    random_state=42,
)
bpr_losses = bpr_model.fit(R_train)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(bpr_losses) + 1), bpr_losses, marker="o")
ax.set_xlabel("Epoch")
ax.set_ylabel("Avg BPR loss (log-likelihood)")
ax.set_title("BPR-MF training curve")
plt.tight_layout()
plt.show()

In [ ]:
metrics_bpr_scratch = evaluate(
    bpr_model.user_factors, bpr_model.item_factors,
    R_train, val_user_items, k=K
)
print("BPR-MF (from scratch):", metrics_bpr_scratch)

---

## 4. Library comparison: regression vs ranking

Let's compare our from-scratch BPR with the `implicit` library's optimized implementations of both ALS and BPR.

In [ ]:
from implicit.bpr import BayesianPersonalizedRanking

# BPR from implicit library
bpr_lib = BayesianPersonalizedRanking(
    factors=N_FACTORS,
    learning_rate=0.01,
    regularization=0.001,
    iterations=15,
    random_state=42,
)
bpr_lib.fit(R_train)

metrics_bpr_lib = evaluate(
    bpr_lib.user_factors, bpr_lib.item_factors,
    R_train, val_user_items, k=K
)
print("BPR (implicit lib):", metrics_bpr_lib)

In [ ]:
from lightfm import LightFM

# WARP from LightFM — samples violating negatives and weights by approximate rank
# max_sampled: how many negatives to try before giving up.
# Default is 10, but with 24% density most samples hit positives — need more.
warp_model = LightFM(
    no_components=N_FACTORS,
    loss="warp",
    learning_rate=0.05,
    item_alpha=0.001,
    user_alpha=0.001,
    max_sampled=50,
    random_state=42,
)

for epoch in tqdm(range(15), desc="WARP epochs"):
    warp_model.fit_partial(R_train, epochs=1)

In [ ]:
# LightFM scores = user_embed @ item_embed + user_bias + item_bias
# Fold biases into embeddings so our evaluate() works:
#   u' = [u_embed, 1, u_bias]   i' = [i_embed, i_bias, 1]
#   u' @ i' = u_embed @ i_embed + i_bias + u_bias
warp_user_factors = np.hstack([
    warp_model.user_embeddings,
    np.ones((n_users, 1), dtype=np.float32),
    warp_model.user_biases.reshape(-1, 1),
])
warp_item_factors = np.hstack([
    warp_model.item_embeddings,
    warp_model.item_biases.reshape(-1, 1),
    np.ones((n_items, 1), dtype=np.float32),
])

metrics_warp = evaluate(
    warp_user_factors, warp_item_factors,
    R_train, val_user_items, k=K
)
print("WARP (LightFM):", metrics_warp)

In [ ]:
# Regression vs Ranking comparison table
comparison_df = pd.DataFrame([
    {"Model": "iALS binary (implicit)", "Type": "Regression", **metrics_binary},
    {"Model": "iALS timespent conf. (implicit)", "Type": "Regression", **metrics_linear},
    {"Model": "iALS log conf. (implicit)", "Type": "Regression", **metrics_log},
    {"Model": "BPR-MF (from scratch)", "Type": "Ranking", **metrics_bpr_scratch},
    {"Model": "BPR-MF (implicit)", "Type": "Ranking", **metrics_bpr_lib},
    {"Model": "WARP (LightFM)", "Type": "Ranking", **metrics_warp},
])
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_names = [f"HitRate@{K}", f"MRR@{K}", f"NDCG@{K}"]
colors = ["#4C72B0" if t == "Regression" else "#DD8452" for t in comparison_df["Type"]]

for ax, metric in zip(axes, metrics_names):
    bars = ax.barh(comparison_df["Model"], comparison_df[metric], color=colors)
    ax.set_xlabel(metric)
    ax.set_title(metric)

# Legend
from matplotlib.patches import Patch
axes[0].legend(
    handles=[Patch(color="#4C72B0", label="Regression"),
             Patch(color="#DD8452", label="Ranking")],
    loc="lower right"
)

plt.tight_layout()
plt.show()

---

## 5. Content ALS

### 5.1 The cold-start problem

Standard ALS/BPR learns latent factors $\mathbf{v}_i$ for each item from interactions. But what about **new items** with zero interactions? Their factors remain at random initialization — the model cannot recommend them.

This is the **cold-start problem**. If we have **content features** (e.g., video embeddings from a visual model), we can use them to predict item factors for items with no interactions.

### 5.2 Content ALS formulation

Instead of learning each item factor $\mathbf{v}_i \in \mathbb{R}^k$ independently, we model it as a **linear projection** of the item's content embedding $\mathbf{e}_i \in \mathbb{R}^d$:

$$
\mathbf{v}_i = W \mathbf{e}_i + \mathbf{b}_i
$$

where:
- $W \in \mathbb{R}^{k \times d}$ is a shared projection matrix
- $\mathbf{b}_i \in \mathbb{R}^k$ is a per-item bias (residual) that captures information not present in content

For items with interactions, $\mathbf{b}_i$ is learned from data. For cold-start items, we set $\mathbf{b}_i = 0$ and use the content projection $\mathbf{v}_i = W \mathbf{e}_i$ as the item factor.

### 5.3 Objective

We use the implicit ALS objective, but with the item factor decomposition $\mathbf{v}_i = W \mathbf{e}_i + \mathbf{b}_i$:

$$
\mathcal{L} = \sum_{u,i} c_{ui} \left(p_{ui} - \mathbf{u}_u^\top (W \mathbf{e}_i + \mathbf{b}_i)\right)^2 + \lambda \left(\|\mathbf{u}_u\|^2 + \|W\|_F^2 + \|\mathbf{b}_i\|^2\right)
$$

### 5.4 Alternating optimization

We optimize three sets of parameters in turn:

**Step 1: Fix $W$, $\mathbf{b}$, update users $U$**

Same as standard iALS — the effective item factor is $\mathbf{v}_i = W\mathbf{e}_i + \mathbf{b}_i$:

$$
\mathbf{u}_u = \left(V^\top V + V^\top (C_u - I) V + \lambda I\right)^{-1} V^\top C_u \mathbf{p}_u
$$

**Step 2: Fix $U$, $W$, update item biases $\mathbf{b}_i$**

For each item $i$, define $\tilde{p}_{ui} = p_{ui} - \mathbf{u}_u^\top W \mathbf{e}_i$ (residual preference after content contribution). Then:

$$
\mathbf{b}_i = \left(U_i^\top C_i U_i + \lambda I\right)^{-1} U_i^\top C_i \tilde{\mathbf{p}}_i
$$

**Step 3: Fix $U$, $\mathbf{b}$, update projection $W$**

Define the residual $\tilde{p}_{ui} = p_{ui} - \mathbf{u}_u^\top \mathbf{b}_i$. We need to solve for $W$ from:

$$
\min_W \sum_{u,i} c_{ui} \left(\tilde{p}_{ui} - \mathbf{u}_u^\top W \mathbf{e}_i\right)^2 + \lambda \|W\|_F^2
$$

Using $\text{vec}(W)$ notation, this is a ridge regression in the Kronecker product space. In practice, we can solve it row by row: each row $\mathbf{w}_l$ of $W$ (the $l$-th latent dimension) satisfies:

$$
\mathbf{w}_l = \left(\sum_{u,i} c_{ui} \cdot u_{ul}^2 \cdot \mathbf{e}_i \mathbf{e}_i^\top + \lambda I\right)^{-1} \sum_{u,i} c_{ui} \cdot u_{ul} \cdot \tilde{p}_{ui} \cdot \mathbf{e}_i
$$

For efficiency, we approximate the $W$ update by treating it as a global regression from content embeddings to the current effective item factors (after subtracting biases).

### 5.5 Content ALS implementation

In [ ]:
class ContentALS:
    """Implicit ALS with content embedding projection for cold-start items.

    Item factors are decomposed as: v_i = W @ e_i + b_i
    where W is a shared projection from content embeddings and b_i is a per-item residual.
    """

    def __init__(self, n_factors=64, n_iters=15, reg=0.01, alpha=1.0,
                 content_reg=0.1, random_state=42):
        self.n_factors = n_factors
        self.n_iters = n_iters
        self.reg = reg
        self.alpha = alpha
        self.content_reg = content_reg
        self.random_state = random_state

        self.user_factors = None   # (n_users, k)
        self.item_biases = None    # (n_items, k)
        self.W = None              # (k, emb_dim) — content projection

    def _item_factors(self, E):
        """Compute effective item factors: V = E @ W^T + B."""
        return E @ self.W.T + self.item_biases

    def fit(self, R_csr, E, verbose=True):
        """Fit Content ALS.

        R_csr: (n_users, n_items) CSR interaction matrix.
        E: (n_items, emb_dim) content embedding matrix.
        """
        n_users, n_items = R_csr.shape
        emb_dim = E.shape[1]
        k = self.n_factors
        rng = np.random.default_rng(self.random_state)

        # Initialize
        self.user_factors = rng.normal(0, 0.01, (n_users, k)).astype(np.float32)
        self.item_biases = np.zeros((n_items, k), dtype=np.float32)
        self.W = rng.normal(0, 0.01, (k, emb_dim)).astype(np.float32)

        R_csc = R_csr.tocsc()
        I_k = np.eye(k, dtype=np.float32)
        I_d = np.eye(emb_dim, dtype=np.float32)

        for it in range(self.n_iters):
            # Effective item factors
            V = self._item_factors(E)

            # --- Step 1: Update user factors (standard iALS) ---
            VtV = V.T @ V  # (k, k) — precompute once
            for u in tqdm(range(n_users),
                          desc=f"ContentALS iter {it+1}/{self.n_iters} [users]",
                          leave=False, disable=not verbose):
                s, e = R_csr.indptr[u], R_csr.indptr[u + 1]
                idx = R_csr.indices[s:e]
                if len(idx) == 0:
                    continue
                conf_vals = self.alpha * R_csr.data[s:e]  # c_ui - 1

                V_u = V[idx]  # (n_u, k)
                # A = V^T V + V_u^T diag(c-1) V_u + lambda I
                A = VtV + V_u.T @ (V_u * conf_vals[:, None]) + self.reg * I_k
                # b = V_u^T @ (c_ui * p_ui) — p_ui = 1 for observed
                b_vec = V_u.T @ (1.0 + conf_vals)
                self.user_factors[u] = np.linalg.solve(A, b_vec)

            # --- Step 2: Update item biases b_i ---
            U = self.user_factors
            UtU = U.T @ U  # (k, k) — precompute once
            content_proj = E @ self.W.T  # (n_items, k) — W contribution

            for i in tqdm(range(n_items),
                          desc=f"ContentALS iter {it+1}/{self.n_iters} [items]",
                          leave=False, disable=not verbose):
                s, e = R_csc.indptr[i], R_csc.indptr[i + 1]
                idx = R_csc.indices[s:e]
                if len(idx) == 0:
                    self.item_biases[i] = 0.0
                    continue
                conf_vals = self.alpha * R_csc.data[s:e]

                U_i = U[idx]  # (n_i, k)
                # Residual preference: p_ui - u_u^T (W e_i)
                content_scores = U_i @ content_proj[i]  # (n_i,)
                residual = (1.0 + conf_vals) - content_scores * (1.0 + conf_vals)
                # Actually: c_ui * (p_ui - u^T W e_i), but p_ui=1 for observed
                # For unobserved, p_ui=0 and c_ui=1, contributing -u^T W e_i to the sum
                # Efficient: use the VtV trick on residuals

                A = UtU + U_i.T @ (U_i * conf_vals[:, None]) + self.reg * I_k
                # b = sum over observed: c_ui * (p_ui - u^T W e_i) * u_u + sum over all: 1 * (0 - u^T W e_i) * u_u
                # = U_i^T (c * 1) - (UtU + U_i^T diag(c-1) U_i) @ content_proj[i]
                # Simplified: solve for b_i in normal equations
                b_target = U_i.T @ (1.0 + conf_vals) - A @ content_proj[i] + self.reg * content_proj[i]
                self.item_biases[i] = np.linalg.solve(A, b_target)

            # --- Step 3: Update W via ridge regression ---
            # Approximate: fit W so that W @ E^T ≈ (V_target - B)^T
            # where V_target are the item factors from a standard iALS step
            # This is: for each latent dim l, solve w_l = argmin ||E w_l - target_l||^2 + reg ||w_l||^2
            V_current = self._item_factors(E)
            # Target for W: items with interactions should have factors close to V_current
            # We use the effective item factors minus biases as the regression target
            target = V_current - self.item_biases  # this should be ≈ E @ W^T

            # But we want W that maps E -> item latent space
            # Ridge: W^T = (E^T E + reg I)^{-1} E^T target
            EtE = E.T @ E + self.content_reg * I_d  # (d, d)
            self.W = np.linalg.solve(EtE, E.T @ target).T  # (k, d)

            if verbose:
                V_final = self._item_factors(E)
                recon = np.sum(V_final ** 2)
                print(f"  Iter {it+1}: ||V||^2 = {recon:.2f}")

        return self

    def get_item_factors(self, E):
        """Get effective item factors for given content embeddings."""
        return self._item_factors(E)

    def get_cold_start_factors(self, E_new):
        """Get item factors for new items using only content projection (b=0)."""
        return E_new @ self.W.T

### 5.6 Train Content ALS on VK-LSVD

In [ ]:
content_als = ContentALS(
    n_factors=N_FACTORS,
    n_iters=10,
    reg=REG,
    alpha=1.0,
    content_reg=0.1,
    random_state=42,
)
content_als.fit(R_train, item_content_emb)

In [ ]:
# Evaluate Content ALS on all items (warm)
V_content = content_als.get_item_factors(item_content_emb)

metrics_content_als = evaluate(
    content_als.user_factors, V_content,
    R_train, val_user_items, k=K
)
print("Content ALS (all items):", metrics_content_als)

### 5.7 Cold-start evaluation

To evaluate cold-start performance, we simulate cold items: take items that appear in the **validation set** but hold them out from training. This guarantees we have ground-truth interactions for these items.

We evaluate in two regimes:
- **Cold-only ranking**: rank only among cold items (isolates cold-start quality)
- **Mixed ranking**: rank among all items (realistic scenario)

In [ ]:
# Simulate cold-start: hold out items that appear in validation and have embeddings
val_item_set = set(val_i_idx.tolist())
train_item_counts = np.diff(R_train.tocsc().indptr)

# Candidate cold items: in validation, have interactions in training, and have content embeddings
candidate_cold = np.array([
    i for i in val_item_set
    if has_embedding[i] and train_item_counts[i] > 0
])

rng_cold = np.random.default_rng(123)
cold_frac = 0.2
cold_items = rng_cold.choice(candidate_cold,
                              size=int(len(candidate_cold) * cold_frac),
                              replace=False)
cold_items_set = set(cold_items.tolist())

print(f"Cold items: {len(cold_items):,} (selected from {len(candidate_cold):,} candidates)")

# Remove cold items from training matrix (vectorized via COO)
R_coo = R_train.tocoo()
keep = ~np.isin(R_coo.col, cold_items)
R_train_warm = sparse.csr_matrix(
    (R_coo.data[keep], (R_coo.row[keep], R_coo.col[keep])),
    shape=R_train.shape,
)

print(f"R_train_warm nnz: {R_train_warm.nnz:,} (was {R_train.nnz:,})")

# Validation ground truth restricted to cold items
val_user_items_cold = {}
for u, items in val_user_items.items():
    cold_gt = [i for i in items if i in cold_items_set]
    if cold_gt:
        val_user_items_cold[u] = cold_gt

print(f"Val users with cold-item ground truth: {len(val_user_items_cold):,}")

In [ ]:
# Cold-only evaluation: rank ONLY among cold items
def evaluate_cold(user_factors, item_factors, cold_item_indices,
                  R_filter, val_user_items, k=10):
    """Evaluate ranking quality considering only cold items as candidates."""
    hits = 0
    mrr_sum = 0.0
    ndcg_sum = 0.0
    n_eval = 0

    for u, true_items in val_user_items.items():
        scores = item_factors @ user_factors[u]

        # Mask out everything except cold items
        mask = np.full(len(scores), -np.inf)
        mask[cold_item_indices] = scores[cold_item_indices]
        scores = mask

        # Also filter items seen in training
        s, e = R_filter.indptr[u], R_filter.indptr[u + 1]
        scores[R_filter.indices[s:e]] = -np.inf

        n_valid = np.sum(np.isfinite(scores))
        actual_k = min(k, n_valid)
        if actual_k == 0:
            continue

        top_k = np.argpartition(-scores, actual_k)[:actual_k]
        top_k = top_k[np.argsort(-scores[top_k])]

        true_set = set(true_items)
        hits += int(bool(true_set & set(top_k)))

        for rank, item in enumerate(top_k, 1):
            if item in true_set:
                mrr_sum += 1.0 / rank
                break

        dcg = sum(1.0 / np.log2(r + 2) for r, item in enumerate(top_k) if item in true_set)
        n_rel = min(k, len(true_items))
        idcg = sum(1.0 / np.log2(i + 2) for i in range(n_rel))
        ndcg_sum += dcg / idcg if idcg > 0 else 0.0

        n_eval += 1

    if n_eval == 0:
        return {f"HitRate@{k}": 0, f"MRR@{k}": 0, f"NDCG@{k}": 0}
    return {
        f"HitRate@{k}": hits / n_eval,
        f"MRR@{k}": mrr_sum / n_eval,
        f"NDCG@{k}": ndcg_sum / n_eval,
    }


# Train Content ALS on warm-only interactions
content_als_cold = ContentALS(
    n_factors=N_FACTORS, n_iters=10, reg=REG, alpha=1.0,
    content_reg=0.1, random_state=42
)
content_als_cold.fit(R_train_warm, item_content_emb)

# For cold items: use content-only factors (b_i = 0)
V_cold = content_als_cold.get_item_factors(item_content_emb)
V_cold_only = V_cold.copy()
V_cold_only[cold_items] = content_als_cold.get_cold_start_factors(item_content_emb[cold_items])

# Train standard iALS on warm-only for comparison
ials_warm = AlternatingLeastSquares(
    factors=N_FACTORS, regularization=REG, iterations=N_ITERS, random_state=42
)
ials_warm.fit(R_train_warm)

# Evaluate on cold items only — ranking among cold candidates only
print("=== Cold-start evaluation (ranking among cold items only) ===")
metrics_content_cold = evaluate_cold(
    content_als_cold.user_factors, V_cold_only, cold_items,
    R_train_warm, val_user_items_cold, k=K
)
print("Content ALS (cold items):", metrics_content_cold)

metrics_ials_cold = evaluate_cold(
    ials_warm.user_factors, ials_warm.item_factors, cold_items,
    R_train_warm, val_user_items_cold, k=K
)
print("iALS (cold items):       ", metrics_ials_cold)

In [ ]:
# Visualize cold-start results
cold_df = pd.DataFrame([
    {"Model": "iALS (no content)", **metrics_ials_cold},
    {"Model": "Content ALS", **metrics_content_cold},
])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, [f"HitRate@{K}", f"MRR@{K}", f"NDCG@{K}"]):
    ax.bar(cold_df["Model"], cold_df[metric], color=["#4C72B0", "#55A868"])
    ax.set_ylabel(metric)
    ax.set_title(f"Cold-start: {metric}")

plt.suptitle("Cold-start item evaluation", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Full comparison & conclusions

In [ ]:
# Full comparison table (warm items — standard evaluation)
final_df = pd.DataFrame([
    {"Model": "iALS binary", "Type": "Regression", **metrics_binary},
    {"Model": "iALS timespent conf.", "Type": "Regression", **metrics_linear},
    {"Model": "iALS log conf.", "Type": "Regression", **metrics_log},
    {"Model": "BPR-MF (scratch)", "Type": "Ranking", **metrics_bpr_scratch},
    {"Model": "BPR-MF (implicit)", "Type": "Ranking", **metrics_bpr_lib},
    {"Model": "WARP (LightFM)", "Type": "Ranking", **metrics_warp},
    {"Model": "Content ALS", "Type": "Regression + Content", **metrics_content_als},
])

print("=== Warm-item evaluation ===")
print(final_df.to_string(index=False))

In [ ]:
# Final visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

color_map = {
    "Regression": "#4C72B0",
    "Ranking": "#DD8452",
    "Regression + Content": "#55A868",
}
colors = [color_map[t] for t in final_df["Type"]]

for ax, metric in zip(axes, [f"HitRate@{K}", f"MRR@{K}", f"NDCG@{K}"]):
    bars = ax.barh(final_df["Model"], final_df[metric], color=colors)
    ax.set_xlabel(metric)
    ax.set_title(metric)

from matplotlib.patches import Patch
axes[0].legend(
    handles=[
        Patch(color="#4C72B0", label="Regression"),
        Patch(color="#DD8452", label="Ranking"),
        Patch(color="#55A868", label="Regression + Content"),
    ],
    loc="lower right"
)

plt.suptitle("Regression vs Ranking vs Content ALS — Warm Items", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Summary

In this seminar we:

1. **Compared regression and ranking losses** for matrix factorization. Regression losses (MSE, WMSE/iALS) optimize pointwise prediction, while ranking losses (BPR) directly optimize pairwise item ordering. We implemented **BPR-MF from scratch** and compared it with iALS variants.

2. **Explored confidence functions** for iALS (binary, linear, log) and their effect on recommendation quality.

3. **Derived and implemented Content ALS** — a modification of iALS that decomposes item factors as $\mathbf{v}_i = W\mathbf{e}_i + \mathbf{b}_i$, using a shared linear projection $W$ from content embeddings and per-item residual biases $\mathbf{b}_i$.

4. **Evaluated cold-start performance**: Content ALS can recommend new items with zero interactions by using their content embeddings ($\mathbf{v}_i = W\mathbf{e}_i$), while standard ALS/BPR cannot.

### When to use which approach

| Scenario | Recommended approach |
|----------|---------------------|
| Warm items, abundant interactions | iALS or BPR — both work well |
| Ranking metrics are critical | BPR — directly optimizes pairwise ordering |
| Cold-start items with content features | Content ALS — leverages content embeddings |
| Mixed warm + cold catalog | Content ALS with biases — best of both worlds |